# 07 — Deduplication

Same mental model as entity resolution—rules, blocking, evaluation, and the measurement loop—but on a **single table**: find rows that represent the same entity (duplicates) even when IDs differ. Matcher gives you `Deduplicator`, which wraps the same ideas so you don't have to learn a new API. We use the sample deduplication set (1000 rows, 50 known duplicate pairs).

**Back:** [06 Blocking](06_blocking.ipynb) · **TOC:** [Preamble & TOC](00_preamble_and_toc.ipynb)

## 1. Load data and ground truth

For deduplication, ground truth is "pairs of row IDs that are the same entity." You might get that from a golden set or from a rule (e.g. same email ⇒ duplicate). **Data:** This set has **1000 rows**: 50 duplicate pairs (same email, two rows each) and 900 unique rows. The loader returns the table and a ground-truth DataFrame of those 50 pairs.

In [ ]:
import sys
from pathlib import Path
import polars as pl
from matcher import Deduplicator

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_deduplication

df, ground_truth = load_deduplication()
print(f"Source: {df.shape[0]} rows")
print(f"Ground truth: {ground_truth.shape[0]} duplicate pairs")
print(ground_truth.head(5))

## 2. Exact matching with Deduplicator

`Deduplicator(source=df, id_col="id")` treats your table as both "left" and "right" under the hood. `match(rules=...)` returns duplicate pairs; self-pairs (same id on both sides) are dropped automatically. Same rules and blocking options you're used to from Matcher.

In [ ]:
deduplicator = Deduplicator(source=df, id_col="id")
results = deduplicator.match(rules="email")
print(f"Duplicate pairs (email rule): {results.count}")
print(results.matches.select(["id", "id_right", "email"]).head(5))

## 3. Evaluate

Same `evaluate()` API as for Matcher. The match table has `id` and `id_right`, so we pass `right_id_col="id_right"`. Then you're back in the same loop: change rules or blocking, re-run, compare metrics.

In [ ]:
metrics = results.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
print(f"Precision: {metrics['precision']:.2%}, Recall: {metrics['recall']:.2%}, F1: {metrics['f1']:.2%}")
print(f"True positives: {metrics['true_positives']}, False positives: {metrics['false_positives']}, False negatives: {metrics['false_negatives']}")

## 4. Optional: fuzzy and blocking

Fuzzy and blocking work the same as in entity resolution: `match_fuzzy(field=..., threshold=...)` and `blocking_key=...` on both `match()` and `match_fuzzy()`. Self-matches are still excluded. So everything you did in 02 (exact), 03 (measurement loop), 04 (fuzzy), 05 (design: combo, blended), and 06 (blocking) transfers—one table instead of two.

In [ ]:
results_fuzzy = deduplicator.match_fuzzy(field="first_name", threshold=0.8)
print(f"Fuzzy duplicate pairs (first_name, 0.8): {results_fuzzy.count}")
results_block = deduplicator.match(rules="email", blocking_key="zip_code")
print(f"Exact with blocking (zip_code): {results_block.count}")

---

**End of tutorial.** You've walked the full path: preparation → exact matching → measurement loop → fuzzy matching → algorithm design (combos, blended) → blocking → deduplication. The habit that ties it together: measure, tune, compare. For more, see the [main README](../../README.md) and the [design docs](../MATCHING_ALGORITHM_DESIGN.md) in `docs/`.